In [23]:
# -*- coding: utf-8 -*-
"""Copy of Museum_Entry.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Ys_PnAPLtYZBrzNAkt93PFDcvEmRRY9O
"""

!pip install wikibaseintegrator # new line of code
import pandas as pd
import re
import time
from wikibaseintegrator.wbi_exceptions import MWApiError, ModificationFailed # 補上這一個
from wikibaseintegrator import WikibaseIntegrator, wbi_login
from wikibaseintegrator import datatypes
from wikibaseintegrator.models import Claims, Sitelinks
from wikibaseintegrator.wbi_config import config
from wikibaseintegrator.models.labels import Labels
from wikibaseintegrator.models.descriptions import Descriptions
from wikibaseintegrator.wbi_exceptions import MWApiError # Import MWApiError
import requests

In [29]:
# 1: Write column Properties to Wikibase
# -----------------------------------------
# find all Pids
df = pd.read_csv('LOD Tablet Dictionary (FG Cuneiform) - museum_FG - LOD Tablet Dictionary (FG Cuneiform) - museum_FG.csv').drop(0)
properties_col_labels = [re.findall(r'P[0-9]+', col)[0] for col in df if col.startswith('P')]
# remove duplicates
properties = list(set(properties_col_labels))
print(properties_col_labels)

['P2', 'P3', 'P287', 'P125', 'P156', 'P146', 'P771', 'P131', 'P131']


### 2: Define Property Mapping

This `property_dict` maps original property IDs (from the source data) to their corresponding new property IDs in the target Wikibase. This is crucial for correctly transferring property values.

In [30]:
# Define the property mapping based on the expected structure.
# This maps original property IDs (from the source CSV) to their target property IDs in your local Wikibase.
property_dict = {
    'P3': 'P8',
    'P146': 'P7',
    'P2': 'P9',
    'P131': 'P4',
    'P156': 'P5',
    'P125': 'P10',
    'P771': 'P6',
    'P287': 'P3' # Note: P287 was previously removed from properties_col_labels, but included here for completeness if needed elsewhere.
}

print("Property Dictionary Defined:")
print(property_dict)

Property Dictionary Defined:
{'P3': 'P8', 'P146': 'P7', 'P2': 'P9', 'P131': 'P4', 'P156': 'P5', 'P125': 'P10', 'P771': 'P6', 'P287': 'P3'}


In [31]:
# Sign in to create WikibaseIntegrator
# when run thru colab: run ngrok http 8181 in the terminal

# config['USER_AGENT'] = 'Property Import Script'
# config['MEDIAWIKI_API_URL'] = 'https://undepressing-vinita-spurtively.ngrok-free.dev'
# WDUSER_test = input("wikibase username")
# WDPASS_test = input("wikibase password")
# login_testwikidata = wbi_login.Login(user=WDUSER_test, password=WDPASS_test, mediawiki_api_url='http://localhost:8181/w/api.php')
config['USER_AGENT'] = 'Property Import Script'

my_ngrok_url = 'https://undepressing-vinita-spurtively.ngrok-free.dev'

WDUSER_test = input("wikibase username: ")
WDPASS_test = input("wikibase password: ")

login_testwikidata = wbi_login.Login(
    user=WDUSER_test,
    password=WDPASS_test,
    mediawiki_api_url=f'{my_ngrok_url}/w/api.php'
)

WDUSER = input("factgrid username")
WDPASS = input("factgrid password")
login_wikidata = wbi_login.Login(user=WDUSER, password=WDPASS, mediawiki_api_url='https://database.factgrid.de/w/api.php') #change to FactGrid's api

wbi = WikibaseIntegrator(login=login_testwikidata)

wikibase username: Aaron1021225
wikibase password: Aaron891016!


factgrid usernameAaron Shen
factgrid passwordnvayk2rM5X7YYZ7


INFO:backoff:Backing off __init__(...) for 1.0s (requests.exceptions.ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))
ERROR:wikibaseintegrator.wbi_backoff:Backing off 1.0 seconds afters 1 tries calling function with args (<wikibaseintegrator.wbi_login.Login object at 0x7bb3e0c72450>,) and kwargs {'user': 'Aaron Shen', 'password': 'nvayk2rM5X7YYZ7', 'mediawiki_api_url': 'https://database.factgrid.de/w/api.php'}


In [26]:
# --- Step 2: Build and Repair Target Entity Mapping ---
target_qid_dict = {}

# 1. Identify all unique FactGrid QIDs used in the property columns (excluding P771)
property_cols_for_mapping = df[[col for col in df if col.startswith('P') and col != 'P771']]
qid_property_cols = df[[i for i in property_cols_for_mapping if (type(property_cols_for_mapping[i][1]) == str and property_cols_for_mapping[i][1].startswith('Q'))]]

qids_list = []
for col in qid_property_cols:
    qids_list += (list(qid_property_cols[col].dropna().unique()))
qids_list = list(set(qids_list))

print(f"🔍 Checking and mapping {len(qids_list)} target entities (cities, orgs, etc.) to local Wikibase...\n")

for qid in qids_list:
    time.sleep(2)  # Increased delay to stabilize ngrok and avoid 503 errors
    try:
        # Fetch the original label from FactGrid
        source_entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
        en_label = str(source_entity.labels.get('en'))

        if not en_label or en_label == 'None':
            en_label = f"Entity {qid}"

        # 2. Search if this label already exists in your local Wikibase
        search_params = {
            'action': 'wbsearchentities',
            'search': en_label,
            'language': 'en',
            'type': 'item',
            'format': 'json'
        }
        search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

        if search_res.get('search'):
            # If it exists locally, just record the mapping (prevents duplicates)
            local_id = search_res['search'][0]['id']
            target_qid_dict[qid] = [local_id]
            print(f"🔗 Mapped existing: {qid} -> {local_id} ({en_label})")
        else:
            # If not found locally, create a new item
            new_item = wbi.item.new()
            new_item.labels.set(language='en', value=en_label)

            # Fetch original description if available
            source_desc = source_entity.descriptions.get('en')
            if source_desc:
                new_item.descriptions.set(language='en', value=str(source_desc))

            new_item.sitelinks = Sitelinks()
            written_item = new_item.write(mediawiki_api_url=f'{my_ngrok_url}/w/api.php', login=login_testwikidata, as_new=True)

            target_qid_dict[qid] = [written_item.id]
            print(f"✅ Created new: {en_label} as {written_item.id} (FactGrid: {qid})")

    except Exception as e:
        print(f"🛑 Error processing {qid}: {e}")

print("\n✨ --- target_qid_dict is now complete --- ✨")

🔍 Checking and mapping 23 target entities (cities, orgs, etc.) to local Wikibase...

🔗 Mapped existing: Q11276 -> Q31 (Access on arrangement with the owners)
🔗 Mapped existing: Q399072 -> Q292 (Special collections library)
🔗 Mapped existing: Q484898 -> Q289 (Art museum)
🔗 Mapped existing: Q220833 -> Q19 (Society)
🔗 Mapped existing: Q11307 -> Q288 (University)
🔗 Mapped existing: Q389595 -> Q28 (Cuneiform Inscriptions Geographical Site Index (CIGS) Version 1.6)
🔗 Mapped existing: Q11250 -> Q275 (Royal Library of Denmark, Copenhagen)
🔗 Mapped existing: Q21411 -> Q283 (Collection)
🔗 Mapped existing: Q145257 -> Q297 (Government agency)
🔗 Mapped existing: Q34987 -> Q290 (Foundation)
🔗 Mapped existing: Q21907 -> Q293 (Company)
🔗 Mapped existing: Q11249 -> Q278 (Library)
🔗 Mapped existing: Q389597 -> Q21 (The FactGrid Cuneiform Project (2021–))
🔗 Mapped existing: Q396070 -> Q282 (Seminary)
🔗 Mapped existing: Q11278 -> Q22 (publicly accessible)
🔗 Mapped existing: Q550435 -> Q277 (University lib

In [33]:
property_dict

{'P3': 'P8',
 'P146': 'P7',
 'P2': 'P9',
 'P131': 'P4',
 'P156': 'P5',
 'P125': 'P10',
 'P771': 'P6',
 'P287': 'P3'}

In [34]:
# 3: Write Entity (sources) + Claims (Property + target) to Wikibasement
source_qids_list = list(df['qid'].dropna().unique())

for i in range(len(source_qids_list)):
    qid = source_qids_list[i]

    # Fetch original entity from FactGrid
    entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
    entity.claims = Claims()
    entity.sitelinks = Sitelinks() # Clear sitelinks to avoid "Unknown site" errors

    # Filter claims for this specific museum
    entry_claims = df[df['qid'] == qid]
    entry_claims = entry_claims[[col for col in entry_claims if col.startswith('P') and col != 'P287']].iloc[0]

    for item_idx in range(len(entry_claims)):
        val = entry_claims.iloc[item_idx] # Using .iloc to avoid FutureWarning
        source_pid = properties_col_labels[item_idx]

        if pd.isna(val):
            continue

        new_pid = property_dict.get(source_pid)
        if not new_pid:
            continue

        # 4. Search and Write (Update logic)
    try:
        # Check if this item already exists locally to get its ID
        search_params = {
            'action': 'wbsearchentities',
            'search': str(entity.labels.get('en')),
            'language': 'en',
            'type': 'item',
            'format': 'json'
        }
        search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

        local_id = None
        if search_res.get('search'):
            local_id = search_res['search'][0]['id']
            entity.id = local_id # Set the ID so WBI knows to UPDATE Q926 instead of creating a new one
            print(f"🔍 Found existing: {local_id}. Preparing update...")

        # Write to local Wikibase
        # as_new=True ONLY if local_id is None
        final_item = entity.write(
            mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
            login=login_testwikidata,
            as_new=not bool(local_id)
        )

        status = "Updated" if local_id else "Created"
        print(f"✅ {status}: {qid} as {final_item.id}")

    except Exception as e:
        print(f"🛑 Failed to write {qid}: {e}")


🔍 Found existing: Q926. Preparing update...
✅ Updated: Q1005358 as Q926
🔍 Found existing: Q927. Preparing update...
✅ Updated: Q1005359 as Q927
🔍 Found existing: Q928. Preparing update...
✅ Updated: Q102010 as Q928
🔍 Found existing: Q929. Preparing update...
✅ Updated: Q1031093 as Q929
🔍 Found existing: Q1427. Preparing update...
✅ Updated: Q1060870 as Q1427
🔍 Found existing: Q1428. Preparing update...
✅ Updated: Q1089449 as Q1428
🔍 Found existing: Q1429. Preparing update...
✅ Updated: Q1349910 as Q1429
🔍 Found existing: Q933. Preparing update...
✅ Updated: Q1349912 as Q933
🔍 Found existing: Q1430. Preparing update...
✅ Updated: Q1349913 as Q1430
🔍 Found existing: Q935. Preparing update...
✅ Updated: Q1349914 as Q935
🔍 Found existing: Q936. Preparing update...
✅ Updated: Q148423 as Q936
🔍 Found existing: Q937. Preparing update...
✅ Updated: Q169485 as Q937
🔍 Found existing: Q938. Preparing update...
✅ Updated: Q172028 as Q938
🔍 Found existing: Q939. Preparing update...
✅ Updated: Q1732

In [35]:
target_qid_dict
# qids_list

{'Q11276': ['Q31'],
 'Q399072': ['Q292'],
 'Q484898': ['Q289'],
 'Q220833': ['Q19'],
 'Q11307': ['Q288'],
 'Q389595': ['Q28'],
 'Q11250': ['Q275'],
 'Q21411': ['Q283'],
 'Q145257': ['Q297'],
 'Q34987': ['Q290'],
 'Q21907': ['Q293'],
 'Q11249': ['Q278'],
 'Q389597': ['Q21'],
 'Q396070': ['Q282'],
 'Q11278': ['Q22'],
 'Q550435': ['Q277'],
 'Q417872': ['Q284'],
 'Q144732': ['Q14'],
 'Q160273': ['Q296'],
 'Q94578': ['Q286'],
 'Q11267': ['Q285'],
 'Q173099': ['Q287'],
 'Q23239': ['Q276']}

In [38]:
properties_col_labels

['P2', 'P3', 'P287', 'P125', 'P156', 'P146', 'P771', 'P131', 'P131']

In [39]:
property_dict

{'P3': 'P8',
 'P146': 'P7',
 'P2': 'P9',
 'P131': 'P4',
 'P156': 'P5',
 'P125': 'P10',
 'P771': 'P6',
 'P287': 'P3'}

In [40]:
property_cols = df[[col for col in df if col.startswith('P')]]
property_cols

,P2,P3,P287,P125,P156,P146,P771,P131,P131.1
1,Q21411,NaN,Q21907,Q11278,https://www.museedelaposte.fr,https://cdli.earth/collections/142,Q2445818,Q389597,Q389595
2,Q23239,NaN,NaN,Q11276,NaN,https://cdli.earth/collections/1059,NaN,Q389597,Q389595
3,Q23239,NaN,Q145257,Q11278,https://www.britishmuseum.org,https://cdli.earth/collections/868,Q6373,Q389597,Q389595
4,Q21411,NaN,Q173099,Q11267,NaN,https://cdli.earth/collections/1274,Q15819388,Q389597,Q389595
5,Q21411,NaN,Q173099,Q11267,NaN,https://cdli.earth/collections/1540,NaN,Q389597,NaN
...,...,...,...,...,...,...,...,...,...
1241,Q23239,NaN,NaN,Q11276,https://turkishmuseums.com/museum/detail/2255-...,NaN,Q28136567,Q389597,Q389595
1242,Q21411,NaN,Q21907,Q11276,https://yushodo.maruzen.co.jp,https://cdli.earth/collections/220,Q11659220,Q389597,Q389595
1243,Q23239,NaN,Q34987,Q11276,https://www.ziegelei-museum.ch,https://cdli.earth/collections/937,Q27479972,Q389597,Q389595
1244,Q21411,NaN,Q173099,Q11267,NaN,https://cdli.earth/collections/985,NaN,Q389597,Q389595


In [41]:
# property_dict
# # properties_col_labels
# properties_col_labels.remove('P287')
# properties_col_labels

# Check if P287 exists before removing to avoid ValueError
if 'P287' in properties_col_labels:
    properties_col_labels.remove('P287')
    print("Successfully removed P287")
else:
    print("P287 was not in the list or already removed")

# Display the resulting dictionary and list
print("Current Property Dictionary:", property_dict)
print("Updated Property Column Labels:", properties_col_labels)

Successfully removed P287
Current Property Dictionary: {'P3': 'P8', 'P146': 'P7', 'P2': 'P9', 'P131': 'P4', 'P156': 'P5', 'P125': 'P10', 'P771': 'P6', 'P287': 'P3'}
Updated Property Column Labels: ['P2', 'P3', 'P125', 'P156', 'P146', 'P771', 'P131', 'P131']


In [42]:
entry_claims

,901
P2,Q23239
P3,NaN
P125,Q11278
P156,https://www.medelhavsmuseet.se
P146,https://cdli.earth/collections/102
P771,Q1331646
P131,Q389597
P131.1,Q389595


In [45]:
source_qids_list = list(df['qid'].dropna().unique())

print("🚀 Starting final import for museum entries...")

for i in range(len(source_qids_list)):
    qid = source_qids_list[i]

    # Fetch original entity from FactGrid
    factgrid_entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)

    # Create a new item for the local Wikibase (target)
    local_entity = wbi.item.new()
    local_entity.labels = Labels() # Initialize empty labels
    local_entity.descriptions = Descriptions() # Initialize empty descriptions
    local_entity.claims = Claims() # Initialize empty claims
    local_entity.sitelinks = Sitelinks() # Explicitly ensure no sitelinks are passed

    # Copy labels and descriptions from FactGrid entity to the new local entity
    if factgrid_entity.labels.get('en'):
        local_entity.labels.set(language='en', value=str(factgrid_entity.labels.get('en')))
    if factgrid_entity.descriptions.get('en'):
        local_entity.descriptions.set(language='en', value=str(factgrid_entity.descriptions.get('en')))

    # Filter claims for this specific museum from the DataFrame
    filtered_entry_claims_series = df[df['qid'] == qid].iloc[0]
    entry_claims_to_process = filtered_entry_claims_series[[col for col in filtered_entry_claims_series.index if col.startswith('P') and col != 'P287']]

    print(f"Processing QID: {qid} (FactGrid) for local item. Claims: {entry_claims_to_process.to_dict()}")

    for source_pid_col, val in entry_claims_to_process.items(): # Iterate over (column_name, value) pairs
        if pd.isna(val):
            continue

        # Extract the base property ID (e.g., 'P131' from 'P131.1')
        source_pid_base = re.findall(r'P[0-9]+', source_pid_col)[0]
        new_pid = property_dict.get(source_pid_base) # Use base ID for mapping
        if not new_pid:
            print(f"  Warning: No mapping found for {source_pid_base} (from column {source_pid_col}). Skipping.")
            continue

        # Ensure value is a string for consistent checking
        val_str = str(val)

        if source_pid_base == 'P771': # FactGrid ID (ExternalID)
            claim = datatypes.ExternalID(prop_nr=new_pid, value = val_str)
            local_entity.claims.add(claim)
            print(f"  Added ExternalID claim: {new_pid} -> {val_str}")
        elif val_str.startswith('Q'): # Item reference
            if val_str in target_qid_dict:
                target_new_id = target_qid_dict[val_str][0]
                claim = datatypes.Item(prop_nr=new_pid, value = target_new_id)
                local_entity.claims.add(claim)
                print(f"  Added Item claim: {new_pid} -> {target_new_id} (from FG {val_str})")
            else:
                print(f"  Warning: Target QID {val_str} not found in target_qid_dict. Skipping claim {source_pid_col}.")
        elif val_str.startswith('http'): # URL
            claim = datatypes.URL(prop_nr=new_pid, value = val_str)
            local_entity.claims.add(claim)
            print(f"  Added URL claim: {new_pid} -> {val_str}")
        elif isinstance(val, str): # General string
            claim = datatypes.String(prop_nr=new_pid, value = val_str)
            local_entity.claims.add(claim)
            print(f"  Added String claim: {new_pid} -> {val_str}")
        else:
            print(f"  Skipping unsupported data type for {source_pid_col}: {type(val)}")

    try:
        # Check if this item already exists locally to get its ID
        search_params = {
            'action': 'wbsearchentities',
            'search': str(local_entity.labels.get('en')),
            'language': 'en',
            'type': 'item',
            'format': 'json'
        }
        # Ensure to use the correct API URL for the local Wikibase
        search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

        local_id = None
        if search_res.get('search'):
            local_id = search_res['search'][0]['id']
            local_entity.id = local_id # Set the ID so WBI knows to UPDATE instead of creating a new one
            print(f"🔍 Found existing: {local_id}. Preparing update...")

        # Write to local Wikibase
        final_item = local_entity.write(
            mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
            login=login_testwikidata,
            as_new=not bool(local_id) # as_new=True if local_id is None, False otherwise
        )

        status = "Updated" if local_id else "Created"
        print(f"✅ {status}: FactGrid QID {qid} as local {final_item.id}")

    except MWApiError as e:
        print(f"🛑 MWApiError writing {qid}: {e}")
    except Exception as e:
        print(f"🛑 Failed to write {qid}: {e}")

    break # Keeping break for now, as it was in the original snippet that broke.

🚀 Starting final import for museum entries...
Processing QID: Q1005358 (FactGrid) for local item. Claims: {'P2': 'Q21411', 'P3': nan, 'P125': 'Q11278', 'P156': 'https://www.museedelaposte.fr', 'P146': 'https://cdli.earth/collections/142', 'P771': 'Q2445818', 'P131': 'Q389597', 'P131.1': 'Q389595'}
  Added Item claim: P9 -> Q283 (from FG Q21411)
  Added Item claim: P10 -> Q22 (from FG Q11278)
  Added URL claim: P5 -> https://www.museedelaposte.fr
  Added URL claim: P7 -> https://cdli.earth/collections/142
  Added ExternalID claim: P6 -> Q2445818
  Added Item claim: P4 -> Q21 (from FG Q389597)
  Added Item claim: P4 -> Q28 (from FG Q389595)
🔍 Found existing: Q926. Preparing update...
✅ Updated: FactGrid QID Q1005358 as local Q926


In [46]:
qid = source_qids_list[21]

entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
entity.claims = Claims()
qid


'Q395501'

In [47]:
entry_claims = df[df['qid'] == qid]
entry_claims = entry_claims[[col for col in entry_claims if col.startswith('P')]].iloc[0]
entry_claims

,22
P2,Q21411
P3,Q529854
P287,Q160273
P125,Q11276
P156,https://www.princeton.edu
P146,NaN
P771,Q21578
P131,Q389597
P131.1,Q389595


In [48]:

target_new_id = target_qid_dict[entry_claims[2]][0]
target_new_id

/tmp/ipykernel_13973/2249257399.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  target_new_id = target_qid_dict[entry_claims[2]][0]


'Q296'

In [49]:
new_pid = property_dict[properties_col_labels[2]]

properties_col_labels
new_pid


'P10'

In [50]:

claim = datatypes.Item(prop_nr=new_pid, value = target_new_id)
entity.claims.add(claim)

<Claims @9d9460 _Claims__claims={'P10': [<Item @658a40 _Claim__mainsnak=<Snak @659040 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P10' _Snak__hash=None _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 296, 'id': 'Q296'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @658b90 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id=None _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @6589e0 _References__references=[]>>]}>

In [51]:
entity

<ItemEntity @82bd70 _BaseEntity__api=<wikibaseintegrator.wikibaseintegrator.WikibaseIntegrator object at 0x7bb3e0ccb590>
	 _BaseEntity__title='Item:Q395501'
	 _BaseEntity__pageid=393509
	 _BaseEntity__lastrevid=16528198
	 _BaseEntity__type='item'
	 _BaseEntity__id='Q395501'
	 _BaseEntity__claims=<Claims @9d9460 _Claims__claims={'P10': [<Item @658a40 _Claim__mainsnak=<Snak @659040 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P10' _Snak__hash=None _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 296, 'id': 'Q296'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @658b90 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id=None _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @6589e0 _References__references=[]>>]}>
	 _ItemEntity__labels=<Labels @5fb950 _LanguageValues__values={'de': <LanguageValue @63bc80 _L

In [53]:
entity.sitelinks = Sitelinks() # Clear sitelinks to avoid 'Unknown site' errors
entity.write(mediawiki_api_url=f'{my_ngrok_url}/w/api.php', login=login_testwikidata, as_new=True)

ERROR:wikibaseintegrator.entities.baseentity:Error while writing to the Wikibase instance
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wikibaseintegrator/entities/baseentity.py", line 287, in _write
    json_result: dict = edit_entity(data=data, id=entity_id, type=self.type, summary=summary, clear=clear, is_bot=is_bot, allow_anonymous=allow_anonymous,
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/wikibaseintegrator/wbi_helpers.py", line 339, in edit_entity
    return mediawiki_api_call_helper(data=params, is_bot=is_bot, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/wikibaseintegrator/wbi_helpers.py", line 221, in mediawiki_api_call_helper
    return mediawiki_api_call('POST', mediawiki_api_url=mediawiki_api_u

ModificationFailed: 'Item [[Item:Q947|Q947]] already has label "Princeton University" associated with language code de, using the same description text.'

In [54]:
#qid ='Q102010'
pid = 'P156'
target = 'https://www.britishmuseum.org/'
entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)

claim = datatypes.URL(prop_nr=pid, value = target)
entity.claims.add(claim)
#entity.write(mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata, as_new=False)

<Claims @5d3da0 _Claims__claims={'P2': [<Item @d394f0 _Claim__mainsnak=<Snak @658c80 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P2' _Snak__hash='2b2c399fae789bd19584ad7e145865e07c148a62' _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 11307, 'id': 'Q11307'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @783530 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id='Q395501$3153f7ad-4c08-4f38-ae56-0bcf3b338503' _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @658560 _References__references=[]>>, <Item @7832f0 _Claim__mainsnak=<Snak @c62cf0 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P2' _Snak__hash='21fcf4d1e448387fbf7f01aafce2837bf49528f0' _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 1188769, 'id': 'Q1188769'}, 'type': 'wikibase-entityid'} _Snak